# Imports

In [ ]:
import os, json
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import wandb
from pathlib import Path
import resnet50_lightning as rn
from datasets import load_dataset

# Constants

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
HF_REPO = "hamzamooraj99/AgriPath-LF16-30k"
ARTIFACT_PATH = "hhm2000-heriot-watt-university/AgriPath-VLM/combined_cnn_paths:v0"
CLASS_NAMES_JSON = None
TOPK = 5 

In [ ]:
wandb.login()

# Load Dataset

In [ ]:
datamodule = rn.AgriPathDataModule(HF_REPO, batch_size=1)
datamodule.setup()

label_idx, idx_label = datamodule.return_labels()
transform = datamodule.transform

num_classes = len(idx_label)

print("Num classes:", num_classes)
print("Example labels:", list(idx_label.items())[:5])

# Load CNN Model from W&B

In [ ]:
def load_cnn_model(artifact_path, exp_idx=0):
    api = wandb.Api()
    artifact = api.artifact(artifact_path, type="model")
    artifact_dir = Path(artifact.download())
    
    # Selecting one of the 9 experiments (e.g., experiment 0)
    checkpoint_path = artifact_dir / f"resnet50_agripath_exp_{exp_idx}.pth"
    learning_rate = 1e-4 
    
    model = rn.ResNet50TLModel(num_classes=65, learning_rate=learning_rate)
    checkpoint = torch.load(checkpoint_path, map_location=torch.device(DEVICE), weights_only=True)
    model.load_state_dict(checkpoint)
    model.to(DEVICE)
    model.eval()
    
    return model

model = load_cnn_model(ARTIFACT_PATH, exp_idx=0)

# Single Inference

## Load Image

In [ ]:
sample = datamodule.test_set[1347]
image = sample['image']
crop = sample['crop']
disease = sample['disease']
crop_disease = sample['crop_disease_label']
print(f"CROP: {crop}")
print(f"DISEASE: {disease}")
display(image)
print(crop_disease)

## Inference

In [ ]:
def predict_sample(sample, model, transform, idx_label, topk=5):
    image = sample['image']

    x = transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        logits = model(x)
        probs = nn.functional.softmax(logits, dim=1)[0]
    
    k = min(topk, probs.numel())
    top_probs, top_idxs = torch.topk(probs, k=k)

    top_labels = [idx_label[int(i)] for i in top_idxs]

    return pd.DataFrame({
        "class": top_labels,
        "prob": top_probs.cpu().numpy()
    })

In [ ]:
df = predict_sample(sample, model, transform, idx_label, TOPK)

df["prob"] = df["prob"].map(lambda x: f"{x:.6f}")

df

# Plot Top-k Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(range(len(df['prob'])), df['prob'])
plt.xticks(range(len(df['prob'])), df['class'], rotation=75, ha="right")
plt.ylabel("Probability")
plt.title(f"Top-{len(df['prob'])} predictions\n{0}")
plt.tight_layout()
plt.show()